# VoiceGuard – AST (Audio Spectrogram Transformer), Model 4

DFADD experiment for Google Colab. Select **Runtime → Change runtime type → T4 GPU** before running the code cell. The notebook converts audio into AST log-Mel inputs, uses a pretrained AST encoder with a frozen backbone, trains a binary classifier, evaluates it, and saves model/results.

In [ ]:
# VOICEGUARD - AST (AUDIO SPECTROGRAM TRANSFORMER) | MODEL 4
# Dataset: DFADD | Google Colab T4 GPU
# Spectrogram patches + pretrained AST + binary classification head

!pip install -q transformers datasets librosa soundfile accelerate scikit-learn

import os, random
import numpy as np
import pandas as pd
import torch
import librosa
from datasets import load_dataset
from transformers import AutoFeatureExtractor, ASTModel
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

# 1. Reproducibility and device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Enable T4 GPU in Runtime > Change runtime type.")

# 2. Load DFADD
print("\nLoading DFADD...")
ds = load_dataset("SpeechAntiSpoofingBenchmarks/DFADD", split="test")
real_ids = [i for i, row in enumerate(ds) if int(row["label"]) == 0]
fake_ids = [i for i, row in enumerate(ds) if int(row["label"]) == 1]
print("Available real:", len(real_ids), "| fake:", len(fake_ids))

# Balanced subset: up to 755 examples from each class
rng = np.random.default_rng(SEED)
n_each = min(755, len(real_ids), len(fake_ids))
selected_ids = np.concatenate([
    rng.choice(real_ids, size=n_each, replace=False),
    rng.choice(fake_ids, size=n_each, replace=False)
])
rng.shuffle(selected_ids)

# 3. Convert audio to mono, 16 kHz, fixed duration
TARGET_SR = 16000
MAX_SECONDS = 5
MAX_LENGTH = TARGET_SR * MAX_SECONDS
audio_data, labels = [], []

print("\nPreprocessing audio...")
for count, idx in enumerate(selected_ids, start=1):
    row = ds[int(idx)]
    audio = np.asarray(row["audio"]["array"], dtype=np.float32)
    sr = int(row["audio"]["sampling_rate"])

    if audio.ndim == 2:
        audio = audio.mean(axis=0) if audio.shape[0] <= 8 else audio.mean(axis=1)

    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
    audio = audio[:MAX_LENGTH]
    if len(audio) < MAX_LENGTH:
        audio = np.pad(audio, (0, MAX_LENGTH - len(audio)))

    audio_data.append(audio.astype(np.float32))
    labels.append(int(row["label"]))

    if count % 100 == 0:
        print(f"Processed {count}/{len(selected_ids)}")

audio_data = np.stack(audio_data)
labels = np.asarray(labels, dtype=np.int64)
print("Audio shape:", audio_data.shape, "| class counts:", np.bincount(labels))

# 4. Train / validation / test split
# This is an exploratory random split of DFADD's evaluation package,
# not an official benchmark split.
X_train, X_temp, y_train, y_temp = train_test_split(
    audio_data, labels, test_size=0.30,
    random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    random_state=SEED, stratify=y_temp
)
print("Train:", len(X_train), "| Validation:", len(X_val), "| Test:", len(X_test))

# 5. Load pretrained AST feature extractor
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
print("\nLoading AST feature extractor:", MODEL_NAME)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

# 6. Dataset: AST extractor creates log-Mel filterbank inputs
class AudioDataset(Dataset):
    def __init__(self, audios, targets):
        self.audios = audios
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        encoded = feature_extractor(
            self.audios[index],
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )
        # AST expects input_values shaped [batch, time, mel_bins].
        features = encoded["input_values"].squeeze(0).float()
        return {
            "input_values": features,
            "label": torch.tensor(self.targets[index], dtype=torch.long)
        }

BATCH_SIZE = 4
train_loader = DataLoader(
    AudioDataset(X_train, y_train),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    AudioDataset(X_val, y_val),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    AudioDataset(X_test, y_test),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

# 7. AST encoder + binary classification head
class VoiceGuardAST(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ASTModel.from_pretrained(MODEL_NAME)

        # Freeze pretrained AST encoder for a resource-efficient baseline
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, input_values):
        with torch.no_grad():
            outputs = self.encoder(input_values=input_values)
            # Exclude the two special tokens; average patch representations
            features = outputs.last_hidden_state[:, 2:, :].mean(dim=1)
        return self.classifier(features)

model = VoiceGuardAST().to(device)

# 8. Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.classifier.parameters(), lr=1e-3, weight_decay=1e-4
)

# 9. Train and validate
EPOCHS = 5
best_val_loss = float("inf")
best_state = None
history = []

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        inputs = batch["input_values"].to(device)
        targets = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    train_loss = total_train_loss / max(1, len(train_loader))

    model.eval()
    total_val_loss = 0.0
    val_probs, val_targets = [], []

    with torch.no_grad():
        for batch in val_loader:
            inputs = batch["input_values"].to(device)
            targets = batch["label"].to(device)
            logits = model(inputs)
            total_val_loss += criterion(logits, targets).item()
            probs = torch.softmax(logits, dim=1)[:, 1]
            val_probs.extend(probs.cpu().numpy())
            val_targets.extend(targets.cpu().numpy())

    val_loss = total_val_loss / max(1, len(val_loader))
    val_predictions = (np.asarray(val_probs) >= 0.5).astype(int)
    val_accuracy = accuracy_score(val_targets, val_predictions)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

if best_state is not None:
    model.load_state_dict(best_state)

# 10. Final test evaluation
model.eval()
test_probs, test_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch["input_values"].to(device)
        targets = batch["label"].to(device)
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1)[:, 1]
        test_probs.extend(probs.cpu().numpy())
        test_targets.extend(targets.cpu().numpy())

test_probs = np.asarray(test_probs)
test_targets = np.asarray(test_targets)
test_predictions = (test_probs >= 0.5).astype(int)

results = pd.DataFrame([{
    "Model": "AST",
    "Accuracy": accuracy_score(test_targets, test_predictions),
    "Precision": precision_score(test_targets, test_predictions, zero_division=0),
    "Recall": recall_score(test_targets, test_predictions, zero_division=0),
    "F1 Score": f1_score(test_targets, test_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(test_targets, test_probs)
}])

print("\n========== AST TEST RESULTS ==========")
print(results.to_string(index=False))
print("\nClassification report:")
print(classification_report(
    test_targets, test_predictions,
    labels=[0, 1], target_names=["Real", "Fake"],
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(test_targets, test_predictions, labels=[0, 1]))

# 11. Save model and results
output_dir = "/content/VoiceGuard_AST"
os.makedirs(output_dir, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(output_dir, "ast_model.pth")
)
results.to_csv(os.path.join(output_dir, "ast_results.csv"), index=False)
pd.DataFrame(history).to_csv(
    os.path.join(output_dir, "ast_training_history.csv"),
    index=False
)

print("\nSaved files in:", output_dir)
print("- ast_model.pth")
print("- ast_results.csv")
print("- ast_training_history.csv")
print("\nExperiment complete. Results are exploratory because this uses a random split of DFADD's evaluation package.")
